# DDPM vs DDIM Sampling Comparison

Simple Colab notebook to:

- clone the repo
- use a checkpoint uploaded into Colab
- generate samples with DDPM and DDIM
- compute FID
- compare sampling speed

Upload these into the Colab Files pane first:

- `best_model.pth`
- `celeba_hq_256` folder OR `celeba_hq_256.zip`


In [ ]:
import shutil
from pathlib import Path

REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH = "main"

WORKDIR = Path("/content/Human-Faces-Generation-Diffusion-Models")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"

print("Project root:", WORKDIR)


## Install dependencies

In [ ]:
!pip install -q torch torchvision tqdm matplotlib scipy

## Prepare dataset

Upload:

- `celeba_hq_256.zip`
or
- extracted `celeba_hq_256` folder


In [ ]:
import zipfile
from pathlib import Path

DATASET_DIR = Path("/content/celeba_hq_256")
ZIP_PATH = Path("/content/celeba_hq_256.zip")

if not DATASET_DIR.exists() and ZIP_PATH.exists():
    print("Extracting dataset zip...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")

print("Dataset path:", DATASET_DIR)
print("Dataset exists:", DATASET_DIR.exists())


## Set paths and experiment settings

In [ ]:
from pathlib import Path

CHECKPOINT_PATH = Path("/content/best_model.pth")

OUT_DIR = Path("/content/ddim_ddpm_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_IMAGES = 100
BATCH_SIZE = 8
FID_BATCH_SIZE = 8

DDIM_STEPS = 50
DDIM_ETA = 0.0

print("Checkpoint:", CHECKPOINT_PATH)
print("Dataset:", DATASET_DIR)
print("Output dir:", OUT_DIR)


## Run DDPM vs DDIM comparison

This saves:

- generated images
- preview grids
- FID CSV
- JSON summary


In [ ]:
%cd /content/Human-Faces-Generation-Diffusion-Models

!python3 scripts/compare_ddim_ddpm_fid.py \
    --checkpoint "$CHECKPOINT_PATH" \
    --dataset_path "$DATASET_DIR" \
    --out_dir "$OUT_DIR" \
    --num_images $NUM_IMAGES \
    --batch_size $BATCH_SIZE \
    --fid_batch_size $FID_BATCH_SIZE \
    --ddim_steps $DDIM_STEPS \
    --ddim_eta $DDIM_ETA \
    --device cuda \
    --fid_device cuda


## Show generated grids

In [ ]:
from IPython.display import Image, display

print("Real test images")
display(Image(filename=str(OUT_DIR / "real_test_grid.png")))

print("DDPM samples")
display(Image(filename=str(OUT_DIR / "ddpm_grid.png")))

print("DDIM samples")
display(Image(filename=str(OUT_DIR / "ddim_grid.png")))


## Show FID results

In [ ]:
import pandas as pd

csv_path = OUT_DIR / "fid_comparison.csv"

df = pd.read_csv(csv_path)

print(df)

df


## Read JSON summary

In [ ]:
import json

summary_path = OUT_DIR / "comparison_summary.json"

with open(summary_path, "r") as f:
    summary = json.load(f)

summary


## Quick notes

- Lower FID is better
- DDIM is usually much faster
- DDPM sometimes gives slightly better quality depending on the checkpoint
